# Neural network with a Lipschitz penalty

Compare GLAd-SSN with Adam on a synthetic regression problem. The objective combines mean squared error with a penalty on sampled input-Hessian norms, estimated by differentiable power iteration.

Reference: [Skip the Hessian, Keep the Rates](https://arxiv.org/abs/2602.08069), November 2026 camera-ready version, Section 5.1 and Appendix E.1, equation (26). Gradient and sampled norm plots correspond to Figures 1 and 2.

Run from the repository root or `NN/`: restart the kernel and **Run All**. Data and initial weights are generated locally; no saved results are required. Set methods, budgets and numerical settings in the first code cell. All methods share the same data and initialization. Outputs are written to a new timestamped directory under `NN/corrected_results/`.

The sampled norm estimate is not a certified global Lipschitz bound. Training uses float64 after float32 data generation and initialization.


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # Windows workaround.

import sys
import importlib
from pathlib import Path
from datetime import datetime
import platform
import pickle
import math
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import functional_call
import matplotlib.pyplot as plt

sys.dont_write_bytecode = True
NN_DIR = Path.cwd()
if not (NN_DIR / "GLAdSSNOptimizer_new.py").exists():
    NN_DIR = NN_DIR / "NN"
sys.path.insert(0, str(NN_DIR.parent.resolve()))
from NN.GLAdSSNOptimizer_new import GlAdSSNOptimizer
from NN import utils_new as nn_utils_module
# Jupyter caches imported modules across cell reruns; load the current helper API.
nn_utils_module = importlib.reload(nn_utils_module)
make_param_vector_interface = nn_utils_module.make_param_vector_interface
hessian_norm_penalty = nn_utils_module.hessian_norm_penalty

# Experiment selection and objective (edit settings here).
RUN_ADAM = True  # False skips Adam; stored results remain available.
GLAD_PERIODS = [ 1, 20 , 50, 100, 200, 400]  # Include 1 explicitly for fresh Hessians; [10] runs only period 10.
N = 100  # Training samples.
penalty_coefficient = 1
no_of_power_iterations = 10  # Same finite-power surrogate for Adam and GLAd.
POWER_EPS = 1e-7  # Current setting; helper default is 1e-12. Changing epsilon changes the objective.

# Training budgets.
SEED, TEST_SEED = 26, 27
ADAM_EPOCHS = 25000
GLAD_EPOCHS = 3000
GRAD_TARGETS = (1e-2, 1e-3, 1e-4)  # Reporting only; these do not stop training.
ADAM_LR = 1e-4
GRAD_TOL = 0.0  # Exact stationarity only; positive values add early stopping.
LAM_MIN = 1e-9  # Numerical damping floor; 0 disables it.
SOLVE_RTOL = 1e-10  # Explicit relative residual; independent of machine epsilon.
MAX_BACKTRACKS = 25  # Up to 26 trials, independent of the Hessian reuse period.
SAVE_RESULTS = True
SAVE_FIGURES = True
DTYPE = torch.float64  # Training and evaluation precision for every method.
INITIALIZATION_DTYPE = torch.float32  # Data and initial-weight precision.
device = torch.device("cpu")
torch.set_default_dtype(DTYPE)
torch.set_num_threads(1)
torch.manual_seed(SEED)
random.seed(SEED)


## Data and network


In [ ]:
X = torch.rand(N, 2, device=device, dtype=INITIALIZATION_DTYPE) * 2 * math.pi
Y = (
    torch.sin(X[:, 0]) * torch.cos(X[:, 1])
    + 0.2 * torch.sin(2 * X[:, 0] + X[:, 1])
    + 0.1 * X[:, 0] * X[:, 1]
)
Y = Y + 0.05 * torch.randn_like(Y)
Y = Y.unsqueeze(1)
# Promote the same rounded data, rather than drawing a different float64 sample.
X, Y = X.to(dtype=DTYPE), Y.to(dtype=DTYPE)

dataset = list(zip(X, Y))
batch_size = len(dataset)  # full batch


class Net(nn.Module):
    def __init__(self, dtype=DTYPE):
        super().__init__()
        self.fc1 = nn.Linear(2, 16, dtype=dtype)
        self.fc2 = nn.Linear(16, 16, dtype=dtype)
        self.fc3 = nn.Linear(16, 1, dtype=dtype)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        return self.fc3(x)


# Snapshot the first network after generating training data (seed 26).
# Every trainer loads these same weights, regardless of its execution order.
initial_state = {
    name: value.detach().to(dtype=DTYPE).clone()
    for name, value in Net(dtype=INITIALIZATION_DTYPE).state_dict().items()
}
all_histories = {}
all_run_info = {}
all_models = {}  # Populated only by the training cells you choose to execute.


## Optional saved results

Keep `LOAD_LONG_RUN=False` for a fresh experiment. To inspect a result saved by this notebook, set `LONG_RUN_FILE` to its path and enable loading. `review` restores and plots saved results without training; `experiment` starts a new comparison from the selected saved weights.


In [ ]:
# Run the imports/settings and data/model setup cells above first.
import copy

LOAD_LONG_RUN = False  # True loads a saved run; False uses the fresh setup above.
LONG_RUN_MODE = "review"  # "review" restores results; "experiment" starts a separate comparison.
WARM_START_METHOD = "GLAd-SSN m=50"  # For experiment mode; None uses the saved original initialization.
LONG_RUN_FILE = None  # Optional path to an exported nn_results_new.pkl.

loaded_long_run = None
loaded_test_dataset = None
long_run_provenance = None
if LOAD_LONG_RUN:
    if LONG_RUN_MODE not in {"review", "experiment"}:
        raise ValueError('LONG_RUN_MODE must be "review" or "experiment"')
    if LONG_RUN_FILE is None:
        raise ValueError("Set LONG_RUN_FILE to a saved result before enabling LOAD_LONG_RUN")
    with Path(LONG_RUN_FILE).open("rb") as stream:
        loaded_long_run = pickle.load(stream)
    saved_metadata = loaded_long_run["metadata"]
    if saved_metadata.get("objective") != "finite_power_surrogate":
        raise ValueError("Load a finite-power result in this notebook")
    if LONG_RUN_MODE == "experiment" and WARM_START_METHOD is not None and WARM_START_METHOD not in loaded_long_run["final_model_states"]:
        raise ValueError(f"Unknown WARM_START_METHOD: {WARM_START_METHOD}")

    if LONG_RUN_MODE == "review":
        # Review uses the settings that generated these histories.
        DTYPE = getattr(torch, saved_metadata["precision"].split(".")[-1])
        INITIALIZATION_DTYPE = getattr(torch, saved_metadata["initialization_precision"].split(".")[-1])
        penalty_coefficient = saved_metadata["penalty_coefficient"]
        no_of_power_iterations = saved_metadata["K"]
        POWER_EPS = saved_metadata["normalization_eps"]
        ADAM_EPOCHS, GLAD_EPOCHS = saved_metadata["adam_epochs"], saved_metadata["glad_epochs"]
        ADAM_LR = saved_metadata["adam_lr"]
        GRAD_TOL, LAM_MIN = saved_metadata["grad_tol"], saved_metadata["lam_min"]
        SOLVE_RTOL, MAX_BACKTRACKS = saved_metadata["solve_rtol"], saved_metadata["max_backtracks"]
        GRAD_TARGETS = tuple(saved_metadata["gradient_targets"])
        # Run All must not repeat the expensive run or export redundant files.
        RUN_ADAM, GLAD_PERIODS = False, []
        SAVE_RESULTS = SAVE_FIGURES = False

    # Both modes reuse the actual saved samples, not newly drawn approximations.
    SEED, TEST_SEED = saved_metadata["seed"], saved_metadata["test_seed"]
    torch.set_default_dtype(DTYPE)
    X = loaded_long_run["training_inputs"].to(device=device, dtype=DTYPE).clone()
    Y = loaded_long_run["training_targets"].to(device=device, dtype=DTYPE).clone()
    N = len(X)
    dataset, batch_size = list(zip(X, Y)), N
    loaded_test_dataset = [(x.to(device=device, dtype=DTYPE).clone(), y.to(device=device, dtype=DTYPE).clone())
                           for x, y in loaded_long_run["test_dataset"]]
    simulated_test_dataset = loaded_test_dataset
    start_method = WARM_START_METHOD if LONG_RUN_MODE == "experiment" else None
    start_weights = (loaded_long_run["initial_state"] if start_method is None
                     else loaded_long_run["final_model_states"][start_method])
    initial_state = {k: v.to(device=device, dtype=DTYPE).clone() for k, v in start_weights.items()}
    all_histories, all_run_info, all_models = {}, {}, {}
    if LONG_RUN_MODE == "review":
        for label, history in loaded_long_run["histories"].items():
            keys = ("loss", "time", "grad_norm", "hessian_norm") + (("Lambda",) if "Lambda" in history else ())
            all_histories[label] = tuple(list(history[key]) for key in keys)
            all_run_info[label] = copy.deepcopy(loaded_long_run["statuses"][label])
            all_run_info[label]["power_iterations"] = saved_metadata["K"]
            all_run_info[label]["precision"] = saved_metadata["precision"]
            model = Net(dtype=DTYPE).to(device)
            model.load_state_dict(loaded_long_run["final_model_states"][label])
            all_models[label] = model
        plot_histories = copy.deepcopy(loaded_long_run["histories"])
        gradient_target_results = copy.deepcopy(loaded_long_run["gradient_targets"])
        run_metadata = copy.deepcopy(saved_metadata)
    else:
        # Keep the settings entered in the first cell; every method shares this start.
        print("Starting a separate comparison from", start_method or "the saved original initialization")
        print("Optimizer state and histories start fresh; the loaded long run remains in loaded_long_run.")
    long_run_provenance = dict(
        file=str(LONG_RUN_FILE.resolve()), mode=LONG_RUN_MODE,
        starting_method=start_method, optimizer_state_restored=False,
        source_metadata=copy.deepcopy(saved_metadata))
    print("Loaded", LONG_RUN_FILE)
    print("Mode:", LONG_RUN_MODE, "| K:", no_of_power_iterations, "| POWER_EPS:", POWER_EPS)
    print("Stored methods:", list(all_histories))
else:
    print("Using the fresh setup above; no saved run loaded.")


def check_stored_power_objective():
    """Do not spend time training before discovering incompatible old results."""
    for label, info in all_run_info.items():
        if (info.get("normalization_eps", 1e-12) != POWER_EPS
            or info.get("penalty_coefficient", penalty_coefficient) != penalty_coefficient
            or info.get("power_iterations", no_of_power_iterations) != no_of_power_iterations
            or info.get("precision", str(DTYPE)) != str(DTYPE)):
            raise ValueError(
                f"{label} uses different objective/precision settings. Set LONG_RUN_MODE='experiment', "
                "rerun this load cell, then rerun the trainer definitions and chosen training cells.")


## Evaluation functions


In [ ]:
def simulate_dataset(N, device=device, noise_std=0.05, generator=None):
    X = torch.rand(N, 2, device=device, dtype=INITIALIZATION_DTYPE, generator=generator) * 2 * math.pi
    Y = (
        torch.sin(X[:, 0]) * torch.cos(X[:, 1])
        + 0.2 * torch.sin(2 * X[:, 0] + X[:, 1])
        + 0.1 * X[:, 0] * X[:, 1]
    )
    Y = Y + noise_std * torch.randn(Y.shape, device=device, dtype=Y.dtype, generator=generator)
    Y = Y.unsqueeze(1)
    return list(zip(X.to(dtype=DTYPE), Y.to(dtype=DTYPE)))


def evaluate_regression_accuracy(model, dataset_to_test, tolerance=0.001, batch_size=None):
    was_training = model.training
    model.eval()

    if batch_size is None:
        batch_size = len(dataset_to_test)

    total_correct = 0.0
    total_sq_error = 0.0
    total_points = 0

    with torch.no_grad():
        for i in range(0, len(dataset_to_test), batch_size):
            batch = dataset_to_test[i : i + batch_size]
            xb = torch.stack([b[0] for b in batch]).to(device)
            yb = torch.stack([b[1] for b in batch]).to(device)

            pred = model(xb)
            err = pred - yb

            total_correct += (err.abs() <= tolerance).float().sum().item()
            total_sq_error += err.pow(2).sum().item()
            total_points += yb.numel()

    if was_training:
        model.train()

    accuracy = total_correct / total_points
    mse = total_sq_error / total_points
    return accuracy, mse

# One independently seeded test set, reused by every method.
simulated_test_dataset = (
    loaded_test_dataset if globals().get("loaded_test_dataset") is not None else
    simulate_dataset(N=1000, generator=torch.Generator(device=device).manual_seed(TEST_SEED)))


def make_start_model():
    model = Net(dtype=DTYPE).to(device)
    model.load_state_dict(initial_state)
    return model.train()


def measure_model(model):
    """Measure loss, full gradient, and max sampled norm at this same point."""
    penalty, max_norm = hessian_norm_penalty(
        model, dict(model.named_parameters()), X, torch.ones_like(X),
        coeff=penalty_coefficient, K=no_of_power_iterations, normalization_eps=POWER_EPS)
    loss = F.mse_loss(model(X), Y) + penalty
    grads = torch.autograd.grad(loss, tuple(model.parameters()))
    grad_norm = torch.cat([g.reshape(-1) for g in grads]).norm()
    return loss.detach().item(), grad_norm.detach().item(), max_norm.detach().item()

## GLAd-SSN trainer


In [ ]:
def train_with_gladssn(
    epochs=50, update_freq=1, lambda0=1.0, m=MAX_BACKTRACKS,
    print_freq=10, return_model=False, symmetrize=True,
):
    model = make_start_model()
    pack, vec_to_param_dict, assign, _ = make_param_vector_interface(model)
    optimizer = GlAdSSNOptimizer(
        model.parameters(), lambda0=lambda0, m=m,
        number_of_steps_to_freeze=update_freq, symmetrize=symmetrize,
        grad_tol=GRAD_TOL, lam_min=LAM_MIN, solve_rtol=SOLVE_RTOL)
    v0 = torch.ones_like(X)

    def loss_from_vec(vec):
        params = vec_to_param_dict(vec)
        penalty, _ = hessian_norm_penalty(
            model, params, X, v0, coeff=penalty_coefficient,
            K=no_of_power_iterations, normalization_eps=POWER_EPS)
        return F.mse_loss(functional_call(model, params, X), Y) + penalty

    t0 = time.perf_counter()
    loss, grad, max_norm = measure_model(model)
    loss_list, grad_list, hessian_norm_list = [loss], [grad], [max_norm]
    time_list, lambda_k_list = [time.perf_counter() - t0], [lambda0]
    info = dict(status="iterations_exceeded", accepted_steps=0, step_diagnostics=[], normalization_eps=POWER_EPS, penalty_coefficient=penalty_coefficient, power_iterations=no_of_power_iterations, precision=str(DTYPE))

    for epoch in range(epochs):
        _, _, next_coefficient = optimizer.step(loss_from_vec, pack, assign)
        status = optimizer.state["status"]
        # Scalar diagnostics only; no additional objective or Hessian evaluations.
        info["step_diagnostics"].append(dict(
            attempted_update=epoch + 1, status=status,
            accepted_damping=optimizer.state["accepted_lam"],
            next_coefficient=optimizer.state["lambda_k"],
            trials=optimizer.state["last_trials"],
            relative_linear_residual=optimizer.state["last_linear_residual"],
            hessian_evaluations=optimizer.state["hessian_evaluations"],
            linear_solves=optimizer.state["linear_solves"]))
        if status != "accepted":
            info["status"] = status
            # Preserve the last accepted point, including time spent on failure.
            time_list[-1] = time.perf_counter() - t0
            break
        loss, grad, max_norm = measure_model(model)
        loss_list.append(loss)
        grad_list.append(grad)
        hessian_norm_list.append(max_norm)
        time_list.append(time.perf_counter() - t0)
        lambda_k_list.append(next_coefficient)  # Lambda, not gradient-scaled lambda.
        if print_freq and (epoch + 1) % print_freq == 0:
            print(f"[GLAd-SSN] {epoch + 1}/{epochs}: loss={loss:.6g}, "
                  f"||g||={grad:.3e}, max sampled norm estimate={max_norm:.4f}")
        if grad <= GRAD_TOL:
            info["status"] = "converged"
            break

    info.update(accepted_steps=optimizer.state["k"],
                hessian_evaluations=optimizer.state["hessian_evaluations"],
                linear_solves=optimizer.state["linear_solves"],
                last_trials=optimizer.state["last_trials"],
                last_linear_residual=optimizer.state["last_linear_residual"],
                last_damping=optimizer.state["accepted_lam"],
                next_coefficient=optimizer.state["lambda_k"], update_freq=update_freq)
    print(f"GLAd-SSN (period {update_freq}): {info['status']}, {info['accepted_steps']} accepted steps")
    result = (loss_list, time_list, grad_list, hessian_norm_list, lambda_k_list, info)
    return (model, *result) if return_model else result

## Adam trainer


In [ ]:
def train_with_adam(
    epochs=20, lr=ADAM_LR, batch_size=None, print_freq=100, return_model=False,
):
    # This notebook compares full-batch methods, as in the paper.
    if batch_size is not None and batch_size != len(dataset):
        raise ValueError("Use full-batch Adam for this comparison")
    model = make_start_model()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=5, threshold=1e-5, min_lr=1e-6)
    t0 = time.perf_counter()
    loss, grad, max_norm = measure_model(model)
    loss_list, grad_list, hessian_norm_list = [loss], [grad], [max_norm]
    time_list = [time.perf_counter() - t0]
    info = dict(status="iterations_exceeded", accepted_steps=0, learning_rates=[], normalization_eps=POWER_EPS, penalty_coefficient=penalty_coefficient, power_iterations=no_of_power_iterations, precision=str(DTYPE))

    for epoch in range(epochs):
        if grad <= GRAD_TOL:
            info["status"] = "converged"
            break
        learning_rate_used = opt.param_groups[0]["lr"]
        opt.zero_grad(set_to_none=True)
        penalty, _ = hessian_norm_penalty(
            model, dict(model.named_parameters()), X, torch.ones_like(X),
            coeff=penalty_coefficient, K=no_of_power_iterations, normalization_eps=POWER_EPS)
        loss_before = F.mse_loss(model(X), Y) + penalty
        loss_before.backward()
        previous = [p.detach().clone() for p in model.parameters()]
        opt.step()
        # Keep the original scheduler's pre-update loss convention.
        scheduler.step(loss_before.detach().item())

        loss, grad, max_norm = measure_model(model)
        if not all(math.isfinite(value) for value in (loss, grad, max_norm)):
            with torch.no_grad():
                for parameter, value in zip(model.parameters(), previous):
                    parameter.copy_(value)
            info["status"] = "nonfinite_objective_or_gradient"
            time_list[-1] = time.perf_counter() - t0
            break
        loss_list.append(loss)
        grad_list.append(grad)
        hessian_norm_list.append(max_norm)
        time_list.append(time.perf_counter() - t0)
        info["accepted_steps"] += 1
        info["learning_rates"].append(learning_rate_used)
        if print_freq and (epoch + 1) % print_freq == 0:
            print(f"[Adam] {epoch + 1}/{epochs}: loss={loss:.6g}, "
                  f"||g||={grad:.3e}, max sampled norm estimate={max_norm:.4f}")

    print(f"Adam: {info['status']}, {info['accepted_steps']} steps")
    result = (loss_list, time_list, grad_list, hessian_norm_list, info)
    return (model, *result) if return_model else result

## Train Adam

Set `RUN_ADAM=False` to skip this baseline.


In [ ]:
# RUN_ADAM is configured in the first code cell.

all_models = globals().get("all_models", {})
if globals().get("LOAD_LONG_RUN", False) and globals().get("LONG_RUN_MODE") == "review":
    print("Review mode: restored Adam result; no training.")
elif RUN_ADAM:
    check_stored_power_objective()
    print("Training with Adam...")
    result = train_with_adam(epochs=ADAM_EPOCHS, lr=ADAM_LR, print_freq=10000, return_model=True)
    model_adam = result[0]
    all_models["Adam"] = model_adam
    all_histories["Adam"] = result[1:-1]
    all_run_info["Adam"] = result[-1]
else:
    print("Skipping Adam; any existing Adam result is retained.")


## Train GLAd-SSN

`GLAD_PERIODS` selects the Hessian update periods. Include `1` for fresh Hessians. Each method starts from the shared initial weights; rerunning a selected method replaces its in-memory result.


In [ ]:
# GLAD_PERIODS is configured in the first code cell.

if any(type(period) is not int or period < 1 for period in GLAD_PERIODS):
    raise ValueError("GLAD_PERIODS must contain positive integers")
if len(GLAD_PERIODS) != len(set(GLAD_PERIODS)):
    raise ValueError("List each GLAd period only once")

# Preserve already trained models, including those in an open pre-change kernel.
all_models = globals().get("all_models", {})
for label, variable in [("Adam", "model_adam"), ("GLAd-SSN", "model_glad")]:
    if label in all_histories and variable in globals():
        all_models.setdefault(label, globals()[variable])
for frequency, previous_model in zip(
    globals().get("list_of_lazy_frequencies", []), globals().get("models_lazy", [])
):
    label = f"GLAd-SSN m={frequency}"
    if label in all_histories:
        all_models.setdefault(label, previous_model)

review_only = globals().get("LOAD_LONG_RUN", False) and globals().get("LONG_RUN_MODE") == "review"
periods_to_train = [] if review_only else GLAD_PERIODS
if review_only:
    print("Review mode: restored GLAd results; no training.")
if periods_to_train:
    check_stored_power_objective()
for period in periods_to_train:
    label = "GLAd-SSN" if period == 1 else f"GLAd-SSN m={period}"
    print("Training with", label)
    result = train_with_gladssn(
        epochs=GLAD_EPOCHS, update_freq=period, print_freq=100, return_model=True)
    all_models[label] = result[0]
    all_histories[label] = result[1:-1]
    all_run_info[label] = result[-1]

print("Stored methods:", list(all_histories))


## Evaluate results

All stored methods must use the same objective and precision.


In [ ]:
# Epsilon, K, coefficient and precision must agree across stored methods.
check_stored_power_objective()

trained_models = [(label, all_models[label]) for label in all_histories]
if not trained_models:
    raise ValueError("No trained models available; run Adam or at least one selected GLAd period")
print("Methods included in evaluation and export:", [label for label, _ in trained_models])


## Export results

Save histories, configuration and final models in a new timestamped directory.


In [ ]:
check_stored_power_objective()  # Also guard direct export-cell reruns.
# New results have their own directory; original data and PDFs are never overwritten.
output_dir = None
if SAVE_RESULTS or SAVE_FIGURES:
    output_dir = NN_DIR / "corrected_results" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    output_dir.mkdir(parents=True, exist_ok=False)

plot_histories = {}
for label, history in all_histories.items():
    saved = {key: list(history[i]) for i, key in enumerate(
        ("loss", "time", "grad_norm", "hessian_norm"))}
    if len(history) > 4:
        saved["Lambda"] = list(history[4])
    plot_histories[label] = saved

# First observed crossing, with no interpolation and no extrapolation.
gradient_target_results = {}
for label, history in plot_histories.items():
    gradient_target_results[label] = {}
    for target in GRAD_TARGETS:
        hit = next((i for i, value in enumerate(history["grad_norm"]) if value <= target), None)
        gradient_target_results[label][f"{target:.0e}"] = dict(
            reached=hit is not None, updates=hit,
            seconds=history["time"][hit] if hit is not None else None)

trained_models = [(label, all_models[label]) for label in all_histories]

run_metadata = dict(
    reference="November 2026 camera-ready version, equation (26), Section 5.1 / Appendix E.1 / Table 3",
    public_reference="https://arxiv.org/abs/2602.08069",
    python=platform.python_version(), torch=torch.__version__, platform=platform.platform(),
    processor=platform.processor(), device=str(device), precision=str(DTYPE),
    threads=torch.get_num_threads(), seed=SEED, test_seed=TEST_SEED,
    initialization_precision=str(INITIALIZATION_DTYPE),
    data_generation="Generate samples, targets and initial weights in float32, then cast to training precision",
    gradient_targets=list(GRAD_TARGETS),
    adam_scheduler="ReduceLROnPlateau per update on pre-update loss; factor .5, patience 5, threshold 1e-5, min_lr 1e-6",
    N=N, noise_std=0.05, architecture=[2,16,16,1], activation="tanh",
    penalty_coefficient=penalty_coefficient, K=no_of_power_iterations,
    normalization_eps=POWER_EPS, objective="finite_power_surrogate",
    adam_epochs=ADAM_EPOCHS, glad_epochs=GLAD_EPOCHS, adam_lr=ADAM_LR,
    grad_tol=GRAD_TOL, lam_min=LAM_MIN, max_backtracks=MAX_BACKTRACKS,
    solve_rtol=SOLVE_RTOL if SOLVE_RTOL is not None else 100*torch.finfo(DTYPE).eps,
    lazy_periods=[all_run_info[label]["update_freq"] for label in all_histories if label != "Adam"],
    requested_glad_periods=list(GLAD_PERIODS),
    adam_included="Adam" in all_histories,
    derivative="Full differentiation through per-sample power iterations",
    norm_metric="Maximum per-sample spectral-norm estimate on training inputs",
    logging="Initial point plus returned iterates; loss, gradient and norm at the same point",
    timing="CPU perf_counter wall time, including diagnostics; model construction excluded")
# Record a loaded comparison's origin instead of claiming newly generated data.
if globals().get("long_run_provenance") is not None:
    run_metadata["loaded_run"] = copy.deepcopy(long_run_provenance)
    run_metadata["data_generation"] = "Reuse saved training/test tensors; cast to recorded training precision"
    run_metadata["initialization"] = (
        "Saved final weights of " + long_run_provenance["starting_method"]
        if long_run_provenance["starting_method"] is not None else "Saved original initial weights")
    if long_run_provenance["starting_method"] is not None:
        run_metadata["initialization_precision"] = str(DTYPE)
    if LONG_RUN_MODE == "review":
        # Historical timing/environment remain those of the loaded run.
        run_metadata = copy.deepcopy(loaded_long_run["metadata"])
        run_metadata["loaded_run"] = copy.deepcopy(long_run_provenance)
if SAVE_RESULTS:
    with (output_dir / "nn_results_new.pkl").open("wb") as result_file:
        pickle.dump(dict(histories=plot_histories, labels=list(all_histories),
                         metadata=run_metadata, statuses=all_run_info,
                         gradient_targets=gradient_target_results,
                         final_model_states={label: {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                                             for label, model in trained_models},
                         initial_state={k:v.cpu() for k,v in initial_state.items()},
                         training_inputs=X.cpu(), training_targets=Y.cpu(),
                         test_dataset=simulated_test_dataset), result_file)
    print("Saved results to", output_dir)
print("Run statuses:", {label: info["status"] for label, info in all_run_info.items()})

## Plot styles


In [ ]:
# Stable colors for the usual methods; extra periods work without a fixed list size.
known_styles = {
    "Adam": ("#E8601C", "None"),
    "GLAd-SSN": ("#9B8AC4", "^"),
    "GLAd-SSN m=5": ("#88A5DD", "D"),
    "GLAd-SSN m=20": ("#8DCBE4", "x"),
    "GLAd-SSN m=50": ("#C2E3D2", "v"),
    "GLAd-SSN m=100": ("#DDECBF", "h"),
    "GLAd-SSN m=1000": ("#F2E1D0", "1"),
}
labels = list(all_histories)
styles = [known_styles.get(label, (plt.get_cmap("tab20")(i % 20), "o"))
          for i, label in enumerate(labels)]
colors = [color for color, marker in styles]
marker_cycle = [marker for color, marker in styles]
line_widths = [1.0] * len(labels)
line_styles = ["-"] * len(labels)


In [ ]:
print(all_histories.keys())



## Gradient and sampled norm plots


In [ ]:
from matplotlib import axes


def plot_graphs(
    list_of_times,
    list_of_grads,
    list_of_labels,
    number_of_methods,
    y_label,
    line_styles,
    which_measure
):
    if number_of_methods is None:
        number_of_methods = len(list_of_times)
    marker_size = 2
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # 2x2 grid
    axes = axes.flatten()  # turn into a simple list: axes[0], axes[1], axes[2], axes[3]
    for i in range(number_of_methods):
        # if labels[i] == "Adam":
            # skip Adam
            # continue
        axes[0].plot(
            list_of_times[i],
            list_of_grads[i],
            label=list_of_labels[i],
            color=colors[i],
            linewidth=1,
            linestyle=line_styles[i],
            # marker=marker_cycle[i],
            markersize=2*marker_size,
            markevery=15000,
        )
    axes[0].set_yscale("symlog", linthresh=1e-12)  # Show exact zeros.
    axes[0].set_xlabel("Runtime (s)")
    axes[0].set_ylabel(y_label)
    axes[0].set_xscale("symlog", linthresh=1)
    axes[0].set_xlim(0, max(max(values) for values in list_of_times))
    # axes[0].legend()
    axes[0].grid(True)

    # Now plot epoch vs grad norm
    # plt.subplot(1, 2, 2)
    for i in range(number_of_methods):
        axes[1].plot(
            list_of_grads[i],
            label=list_of_labels[i],
            color=colors[i],
            # linewidth=line_widths[i],
            linewidth=1,
            linestyle=line_styles[i],
            marker=marker_cycle[i],
            markersize=marker_size,
            markevery=15000,
        )
    axes[1].set_yscale("symlog", linthresh=1e-12)
    axes[1].set_xlabel("Accepted update (initial point = 0)")
    axes[1].set_ylabel(y_label)
    axes[1].set_xscale("symlog", linthresh=1)
    axes[1].set_xlim(0, max(len(values) - 1 for values in list_of_grads))
    # Shared legend
    handles, legend_labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, legend_labels, loc="upper center", ncol=3)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if SAVE_FIGURES:
        plt.savefig(output_dir / ("plot_NN_new_" + which_measure + ".pdf"))
    plt.show()
    plt.close()

def plot_penalty(
    list_of_times,
    list_of_grads,
    list_of_labels,
    number_of_methods,
    y_label,
    line_styles,
    which_measure
):
    if number_of_methods is None:
        number_of_methods = len(list_of_times)
    marker_size = 2
    plt.figure(figsize=(6, 4))

    for i in range(number_of_methods):
        plt.plot(
            list_of_grads[i],
            label=list_of_labels[i],
            color=colors[i],
            # linewidth=line_widths[i],
            linewidth=1,
            linestyle=line_styles[i],
            # marker=marker_cycle[i],
            markersize=marker_size,
            markevery=15000,
        )
    plt.xscale("symlog", linthresh=1)
    plt.xlim(0, max(len(values) - 1 for values in list_of_grads))
    plt.ylim(0, 1.05 * max(1.0, max(max(values) for values in list_of_grads)))
    plt.axhline(1.0, color="black", linestyle="--", linewidth=0.8, label="Penalty threshold")
    plt.xlabel("Accepted update (initial point = 0)")
    plt.ylabel(y_label)
    plt.grid(True, which="both", linestyle="--", linewidth=0.5)
    plt.legend()
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(output_dir / ("plot_NN_new_" + which_measure + ".pdf"))
    plt.show()
    plt.close()
all_methods = list(all_histories.keys())
times_list = [all_histories[m][1] for m in all_methods]
grads_list = [all_histories[m][2] for m in all_methods]
hess_op_norm_list = [all_histories[m][3] for m in all_methods]
lambdas_list = [all_histories[m][4] for m in all_methods if m != "Adam"]

without_adam_times_list = [all_histories[m][1] for m in all_methods if m != "Adam"]


plot_graphs(
    times_list,
    grads_list,
    all_methods,
    None,
    "Gradient norm",
    line_styles,
    which_measure="grad_norm",
)
plot_penalty(
    times_list,
    hess_op_norm_list,
    all_methods,
    None,
    "Max sampled Hessian norm estimate",
    line_styles,
    which_measure="hess_op_norm",
)


## Training and test errors

MSE excludes the penalty. Accuracy is the fraction of predictions within 0.10 of the target. Runtime includes training diagnostics and excludes this final evaluation.


In [ ]:
from IPython.display import Markdown, display

comparison_models = [(label, all_models[label]) for label in all_histories]

comparison_results = {}
table_lines = [
    "| Method | Train MSE | Test MSE | Train accuracy | Test accuracy | Runtime (s) |",
    "| :--- | ---: | ---: | ---: | ---: | ---: |",
]
for method, trained_model in comparison_models:
    train_accuracy, train_mse = evaluate_regression_accuracy(
        trained_model, dataset, tolerance=0.10, batch_size=batch_size)
    test_accuracy, test_mse = evaluate_regression_accuracy(
        trained_model, simulated_test_dataset, tolerance=0.10, batch_size=batch_size)
    runtime_s = all_histories[method][1][-1]  # Last cumulative training time.
    comparison_results[method] = dict(
        train_mse=train_mse, test_mse=test_mse,
        train_accuracy=train_accuracy, test_accuracy=test_accuracy, runtime_s=runtime_s)
    table_lines.append(
        f"| {method} | {train_mse:.6f} | {test_mse:.6f} | "
        f"{train_accuracy:.1%} | {test_accuracy:.1%} | {runtime_s:.2f} |")

display(Markdown("\n".join(table_lines)))

## Time to gradient targets

Report the first recorded crossing as seconds / accepted updates. These reporting targets do not change stopping criteria; a first crossing need not persist.


In [ ]:
target_columns = [f"{target:.0e}" for target in GRAD_TARGETS]
target_lines = [
    "| Method | " + " | ".join(target_columns) + " |",
    "| :--- | " + " | ".join(["---:"] * len(target_columns)) + " |",
]
for label, entries in gradient_target_results.items():
    values = []
    for column in target_columns:
        entry = entries[column]
        values.append(f"{entry['seconds']:.2f} s / {entry['updates']}" if entry['reached'] else "not reached")
    target_lines.append("| " + label + " | " + " | ".join(values) + " |")
display(Markdown("\n".join(target_lines)))
